In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
from sklearn.model_selection import KFold


from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

from tqdm import tqdm


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import os
# Task 1: Write your code here:
data_path = os.path.join(path, 'Q1_data.csv')



data = pd.read_csv(data_path)


In [ ]:
# Task 2: Write your code here:
data.head()

In [ ]:
# Task 3: Write your code here:
data.info()

In [ ]:
# Task 4: Write your code here:
data.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(data['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
data = data.drop(columns=['Order_ID'])
data.info()

In [ ]:
# Task 2: Write your code here:
missing_percentage = (data.isnull().sum() / len(data)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:
#target and key features are missing values, we drop rows
data = data.dropna(subset=['Delivery_Time', 'Weather', 'Traffic_Level','Time_of_Day','Courier_Experience_yrs'])

#checking for missing values again
missing_percentage = (data.isnull().sum() / len(data)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] == 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)


In [ ]:
# Task 3: Write your code here:
#checking for dups
data.duplicated()

In [ ]:
data = data.drop_duplicates()
#verifying
data.duplicated()

In [ ]:
# Task 4: Write your code here:

categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))

data.head()


In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

features = data.columns.drop("Delivery_Time")
scaler = StandardScaler()
data[features] = scaler.fit_transform(data[features])
data.head()

In [ ]:
# Task 6: Write your code here:
#target imbalance is not needed


In [ ]:
def gradient_descent(X, y, learning_rate, n_iters=500):
  m, n = X.shape  # m rows, n columns (dimensions)
  theta = np.zeros(n)  # initialize a zeros weight vector with n dimensions
  losses = []

  for _ in tqdm(range(n_iters), desc="Training Linear Regression"):
    y_hat = np.dot(X, theta)
    gradient = np.dot(X.T, (y_hat - y)) / m
    theta -= learning_rate * gradient

    loss = mean_squared_error(y, y_hat)
    losses.append(loss)

  return theta, losses

In [ ]:
# Task 1: Write your code here:
X  = data[features]
y = data['Delivery_Time']




In [ ]:
# Task 2,3,4,5: Write your code here:

n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
# Storage for linear regression results for each fold
lr_losses = []
lr_mae = []


In [ ]:
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  theta, losses = gradient_descent(X_train.values, y_train.values, learning_rate=0.1, n_iters=500)

  # Validate
  y_pred = np.dot(X_test.values, theta)

  # Calculate evaluation metrics
  mae = mean_absolute_error(y_test, y_pred)
  # Store results
  lr_losses.append(losses)
  lr_mae.append(mae)






In [ ]:

model = RandomForestRegressor(n_estimators=100, max_depth=15,
                             random_state=42)

model.fit(X_train,y_train)

y_pred = model.predict(X_test)




In [ ]:
average_losses = np.mean(lr_losses, axis=0)

plt.figure(figsize=(10, 6))
plt.plot(average_losses, label='Average Loss')
plt.xlabel('Iteration')
plt.ylabel('MSE Loss')
plt.title('Linear Regression Training Loss (Averaged Across Folds)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Task 1: Write your code here:
importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: